# Session 4 CI/CD · The infrastructure pipeline, worked

Two Cloud Storage buckets delivered through plan, policy, approval, apply and drift detection on GitHub Actions.

In [ ]:
source ~/dsba6190-live-demo-04-cicd/env.sh && cd "$WORKDIR" && echo "$PROJECT" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

In [ ]:
# find_run WORKFLOW [COMMIT] [NEWER_THAN]  prints the run id once GitHub has created the run
find_run() {
  local start=$SECONDS id args=(--workflow "$1" --limit 1 --json databaseId)
  [ -n "${2:-}" ] && args+=(--commit "$2")
  echo "waiting for the $1 run to appear" >&2
  until id=$(gh run list "${args[@]}" --jq ".[] | select(.databaseId > ${3:-0}) | .databaseId") && [ -n "$id" ]; do
    if (( SECONDS - start > 120 )); then echo "No $1 run after 120 s. Look at the Actions tab." >&2; return 1; fi
    sleep 3
  done
  echo "$id"
}

# wait_run RUN_ID  returns when the run completes; says once when it waits at the gate
wait_run() {
  local start=$SECONDS told="" status=""
  until status=$(gh run view "${1:?}" --json status --jq .status) && [ "$status" = completed ]; do
    if [ "$status" = waiting ] && [ -z "$told" ]; then
      echo "Run $1 is waiting at the production gate. In the browser: Review deployments, tick production, Approve and deploy."
      told=1
    fi
    if (( SECONDS - start > 900 )); then echo "Run $1 still $status after 15 minutes. Look at it in the browser."; return 1; fi
    sleep 5
  done
  gh run view "$1" --json conclusion,url --jq '"run \(.conclusion) in \(.url)"'
}
echo "helpers loaded"

## Step 1 · The repository, and who it trusts

In [ ]:
gh variable list

In [ ]:
gcloud iam workload-identity-pools providers describe github \
  --workload-identity-pool github-actions --location global \
  --project "$PROJECT" --format "value(attributeCondition)"

## Step 2 · A compliant pull request

In [ ]:
URL=$(gh pr create --base main --head demo/label-cost-center --fill) && PR=${URL##*/} && echo "pull request $PR  $URL"
gh pr view "${PR:?}" --web

In [ ]:
RUN=$(find_run pull-request "$(gh pr view "${PR:?}" --json headRefOid --jq .headRefOid)") && wait_run "$RUN"
gh pr view "$PR" --json comments \
  --jq '.comments[] | select(.body | contains("Terraform plan for")) | .body' \
  | sed '/<details>/,$d'

## Step 3 · Merge, and meet the gate

In [ ]:
gh pr merge "${PR:?}" --squash --delete-branch

In [ ]:
RUN=$(find_run apply "$(gh pr view "${PR:?}" --json mergeCommit --jq .mergeCommit.oid)") && gh run view "$RUN" --web

In [ ]:
wait_run "${RUN:?}"

In [ ]:
gcloud storage buckets describe "gs://$SCRATCH_BUCKET" --format "yaml(labels)"

## Step 4 · A public bucket

In [ ]:
URL=$(gh pr create --base main --head demo/make-scratch-public --fill) && PR=${URL##*/} && echo "pull request $PR  $URL"
gh pr view "${PR:?}" --web

In [ ]:
RUN=$(find_run pull-request "$(gh pr view "${PR:?}" --json headRefOid --jq .headRefOid)") && wait_run "$RUN"
gh pr view "$PR" --json comments \
  --jq '.comments[] | select(.body | contains("Terraform plan for")) | .body' \
  | sed '/<details>/,$d'

In [ ]:
gh pr close "${PR:?}" --delete-branch

## Step 5 · The replace nobody announced

In [ ]:
URL=$(gh pr create --base main --head demo/move-scratch --fill) && PR=${URL##*/} && echo "pull request $PR  $URL"
gh pr view "${PR:?}" --web

In [ ]:
RUN=$(find_run pull-request "$(gh pr view "${PR:?}" --json headRefOid --jq .headRefOid)") && wait_run "$RUN"
gh pr view "$PR" --json comments \
  --jq '.comments[] | select(.body | contains("Terraform plan for")) | .body' \
  | sed '/<details>/,$d'

In [ ]:
gh pr close "${PR:?}" --delete-branch

## Step 6 · Drift

In [ ]:
# The same label edit the runbook makes in the Console
gcloud storage buckets update "gs://$SCRATCH_BUCKET" --update-labels env=prod-by-accident --project "$PROJECT"

In [ ]:
BEFORE=$(gh run list --workflow drift --limit 1 --json databaseId --jq '.[0].databaseId // 0')
gh workflow run drift
RUN=$(find_run drift "" "$BEFORE") && wait_run "$RUN"
gh issue list --label drift --state all --limit 3

In [ ]:
gcloud storage buckets update "gs://$SCRATCH_BUCKET" --update-labels env=dev --project "$PROJECT"

In [ ]:
BEFORE=$(gh run list --workflow drift --limit 1 --json databaseId --jq '.[0].databaseId // 0')
gh workflow run drift
RUN=$(find_run drift "" "$BEFORE") && wait_run "$RUN"
gh issue list --label drift --state all --limit 3

## After class

In [ ]:
cd "${WORKDIR:?}"
git checkout main && git pull
sed -i '' '/prevent_destroy = true/s/true/false/' main.tf

In [ ]:
terraform init -reconfigure \
  -backend-config="bucket=${STATE_BUCKET:?}" \
  -backend-config="prefix=${STATE_PREFIX:?}"

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
git checkout -- main.tf

In [ ]:
# The daily drift plan would report the destroyed buckets every morning. live-setup.sh enables it again.
gh workflow disable drift

In [ ]:
gcloud storage ls --project "$PROJECT" | grep cicd
gh pr list --state open
gh issue list --label drift --state open
git ls-remote --heads origin